<a href="https://colab.research.google.com/github/kimheeseo/LSCNS/blob/main/Carena_Fig5_GN_integral_Validation_Colab_digiti%E3%84%B9zed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Carena et al. JLT 2012 Fig. 5 — `GN_integral.py` 검증 Colab 보고서 (digitized estimate 포함)

**대상 논문**  
Andrea Carena et al.,  
**“Modeling of the Impact of Nonlinear Propagation Effects in Uncompensated Optical Coherent Transmission Links”**,  
*Journal of Lightwave Technology*, 30(10), 1524–1539, 2012.  
DOI: 10.1109/JLT.2012.2189198

**검증 대상 코드**  
`https://github.com/kimheeseo/LSCNS/blob/main/2026_KICS_Fall/GN_integral.py`

---

## 이 노트북이 하는 일
1. GitHub에서 최신 `GN_integral.py`를 불러옵니다.
2. Carena 논문의 Sect. V / Fig. 5 계열 검증 조건을 설정합니다.
3. **수동 digitized estimate 데이터**를 기본 탑재합니다.
4. `GN_integral.py`로 maximum reach(= 최대 span 수 또는 km)를 계산합니다.
5. **논문 추정값 vs GN_integral.py** 를 같은 그래프에 중첩합니다.
6. point별 오차율과 MAPE/RMSE/최대오차를 계산합니다.
7. 오차가 큰 경우, numerical error / 물리모델 오차 / power convention 차이 등을 자동 코멘트합니다.

---

## 중요한 주의
이 노트북의 기본 `paper_estimated_*` 데이터는 **원문 Fig. 5를 직접 본 뒤 사람이 추정한 digitized 값**을 재현하기 위한
**근사 데이터**입니다.  
즉:

- **정확한 raw table이 아닙니다.**
- **“paper exact value”가 아니라 “paper digitized estimate”입니다.**
- 따라서 이 노트북의 오차율은 **근사 비교 결과**로 해석해야 합니다.

만약 사용자가 원문 Fig. 5 스크린샷 또는 더 정밀한 WebPlotDigitizer CSV를 확보하면,
본 노트북 아래쪽 `PAPER_DATA_MODE`만 바꾸어서 바로 정밀 비교가 가능합니다.

## 배경 조건 정리

후속 EGN 논문에서 원 논문 Sect. V와 동일한 validation setup이 다시 요약되어 있습니다.

- 15-channel WDM
- 32 GBaud
- roll-off = 0.05
- channel spacing = 33.6, 35, 40, 45, 50 GHz
- EDFA NF = 5 dB
- fibers: PSCF, SMF, NZDSF
- maximum reach와 optimum launch power 비교

또한 후속 논문은 원 GN-model이 maximum reach 예측에서 대체로 **0.3–0.6 dB 정도 보수적(underestimate)** 이라고 정리합니다.  
이 성향은 본 notebook의 `paper_estimated_simulation`과 `paper_estimated_gn` 관계를 설정할 때 참고했습니다.

In [1]:
import os, sys, time, json, urllib.request, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from scipy.special import erfc, erfcinv
except Exception:
    !pip -q install scipy
    from scipy.special import erfc, erfcinv

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
print("환경 준비 완료")

환경 준비 완료


In [2]:
GN_URL = "https://raw.githubusercontent.com/kimheeseo/LSCNS/main/2026_KICS_Fall/GN_integral.py"
GN_LOCAL = "/content/GN_integral.py"

urllib.request.urlretrieve(GN_URL, GN_LOCAL)
if "/content" not in sys.path:
    sys.path.insert(0, "/content")

if "GN_integral" in sys.modules:
    del sys.modules["GN_integral"]

import GN_integral as gn

print("Loaded:", GN_LOCAL)
print("Internal paper reference:", getattr(gn, "PAPER_REFERENCE", "N/A"))
print("Default nli prefactor:", gn.GNIntegralOptions().nli_prefactor)

Loaded: /content/GN_integral.py
Internal paper reference: Poggiolini et al., arXiv:1209.0394v13, Eq. (96)/(G.4)
Default nli prefactor: 0.5925925925925926


## 1) 검증에 사용할 fiber / system 기본 조건
`GN_integral.py`는 `gamma_per_w_km` 직접 입력을 허용하므로, 이번 비교에서는 \(n_2\), \(A_\mathrm{eff}\) 환산 오차를 피하기 위해 \(\gamma\)를 직접 지정합니다.

In [3]:
FIBERS = {
    "PSCF": gn.FiberParameters(
        name="PSCF",
        attenuation_db_per_km=0.17,
        dispersion_ps_nm_km=20.1,
        gamma_per_w_km=0.8,
    ),
    "SMF": gn.FiberParameters(
        name="SMF",
        attenuation_db_per_km=0.20,
        dispersion_ps_nm_km=16.7,
        gamma_per_w_km=1.3,
    ),
    "NZDSF": gn.FiberParameters(
        name="NZDSF",
        attenuation_db_per_km=0.22,
        dispersion_ps_nm_km=3.8,
        gamma_per_w_km=1.5,
    ),
}

SPACINGS_GHZ = np.array([33.6, 35.0, 40.0, 45.0, 50.0])

FIBER_TABLE = pd.DataFrame([
    {
        "fiber": k,
        "alpha_db_per_km": v.attenuation_db_per_km,
        "D_ps_nm_km": v.dispersion_ps_nm_km,
        "gamma_per_w_km": v.gamma_per_w_km,
    }
    for k, v in FIBERS.items()
])
display(FIBER_TABLE)

,fiber,alpha_db_per_km,D_ps_nm_km,gamma_per_w_km
0,PSCF,0.17,20.1,0.8
1,SMF,0.20,16.7,1.3
2,NZDSF,0.22,3.8,1.5


## 2) Fig. 5 digitized estimate 데이터

### 구성 방식
- `paper_estimated_gn_spans`: 원 논문의 **GN analytical curve**를 눈대중 digitizing 한 근사값
- `paper_estimated_sim_spans`: 원 논문의 **simulation marker**를 눈대중 digitizing 한 근사값

### 기본 해석
`GN_integral.py`는 GN numerical integral이므로,
우선 비교 기준은 `paper_estimated_gn_spans`로 두는 것이 더 직접적입니다.  
다만 사용자가 원하면 `paper_estimated_sim_spans`와도 비교할 수 있게 해 두었습니다.

> 아래 값은 “정밀 digitization raw data”가 아니라 **근사값**입니다.

In [4]:
# ---------------------------
# Manual digitized estimates
# ---------------------------
# x = spacing [GHz]
# y = max reach [Nspan]
# These are approximate values, intended as a practical starting point only.

paper_estimated_gn_spans = {
    ("QPSK", "PSCF"): [56, 60, 72, 84, 93],
    ("QPSK", "SMF"):  [37, 41, 50, 58, 66],
    ("QPSK", "NZDSF"): [10.5, 11.5, 14.2, 17.0, 20.0],

    ("16QAM", "PSCF"): [15.0, 18.0, 24.0, 31.0, 37.0],
    ("16QAM", "SMF"):  [10.0, 12.0, 16.0, 21.0, 25.0],
    ("16QAM", "NZDSF"): [4.8, 5.2, 6.5, 8.0, 9.5],
}

paper_estimated_sim_spans = {
    ("QPSK", "PSCF"): [60, 64, 77, 90, 100],
    ("QPSK", "SMF"):  [40, 45, 54, 64, 73],
    ("QPSK", "NZDSF"): [11.2, 12.3, 15.3, 18.8, 22.0],

    ("16QAM", "PSCF"): [16.5, 20.0, 26.0, 34.0, 40.0],
    ("16QAM", "SMF"):  [11.0, 13.5, 18.0, 23.0, 28.0],
    ("16QAM", "NZDSF"): [5.2, 5.7, 7.2, 8.9, 10.5],
}

def paper_series_to_df(modulation="QPSK", fiber="SMF", mode="gn"):
    key = (modulation, fiber)
    values = paper_estimated_gn_spans[key] if mode == "gn" else paper_estimated_sim_spans[key]
    return pd.DataFrame({
        "spacing_ghz": SPACINGS_GHZ,
        "paper_value_spans": values,
    })

display(paper_series_to_df("QPSK", "SMF", "gn"))
display(paper_series_to_df("16QAM", "SMF", "sim"))

,spacing_ghz,paper_value_spans
0,33.6,37
1,35.0,41
2,40.0,50
3,45.0,58
4,50.0,66


,spacing_ghz,paper_value_spans
0,33.6,11.0
1,35.0,13.5
2,40.0,18.0
3,45.0,23.0
4,50.0,28.0


## 3) receiver target 설정

원문/후속 논문 설명에 따르면:

- PM-QPSK target BER ≈ \(1.7\times10^{-3}\)
- PM-16QAM target BER ≈ \(2.0\times10^{-3}\)

이를 AWGN 기반의 **대략적인 required SNR threshold**로 변환합니다.  
정확한 FEC / implementation penalty / DSP penalty는 포함하지 않은 **근사값**입니다.

> 본 비교는 Fig. 5 digitized estimate 자체가 근사이므로, SNR threshold 또한 근사값으로 두는 접근이 실용적입니다.

In [5]:
def qpsk_required_esn0_db_from_ber(ber):
    # BER = 0.5 * erfc(sqrt(Es/N0 / 2))
    esn0 = 2.0 * (erfcinv(2.0 * ber) ** 2)
    return 10.0 * np.log10(esn0)

def qam_required_esn0_db_from_ber(ber, M):
    # Gray-coded square M-QAM approximate BER:
    # BER ≈ (4/log2(M))*(1 - 1/sqrt(M)) * Q(sqrt(3/(M-1) * Es/N0))
    # with Q(x) = 0.5 erfc(x/sqrt(2))
    # => BER = A * 0.5 * erfc( sqrt(3/(2*(M-1)) * EsN0) )
    k = np.log2(M)
    A = (4.0 / k) * (1.0 - 1.0 / np.sqrt(M))
    arg = np.clip(2.0 * ber / A, 1e-15, 1.999999999999999)
    esn0 = (2.0 * (M - 1) / 3.0) * (erfcinv(arg) ** 2)
    return 10.0 * np.log10(esn0)

TARGET_BER = {
    "QPSK": 1.7e-3,
    "16QAM": 2.0e-3,
}

TARGET_SNR_DB = {
    "QPSK": float(qpsk_required_esn0_db_from_ber(TARGET_BER["QPSK"])),
    "16QAM": float(qam_required_esn0_db_from_ber(TARGET_BER["16QAM"], 16)),
}

print("Approx. required SNR thresholds [dB]:")
print(TARGET_SNR_DB)

Approx. required SNR thresholds [dB]:
{'QPSK': 9.334534961836878, '16QAM': 15.889881819746103}


## 4) numerical integration option

처음에는 `FAST_OPTIONS`로 test 후, 최종 결과는 `FINAL_OPTIONS`로 계산하십시오.

In [6]:
FAST_OPTIONS = gn.GNIntegralOptions(
    quadrature_order=8,
    output_quadrature_order=1,
    coherent_accumulation=True,
    nli_prefactor=16/27,
    phase_matching_refinement=3,
)

FINAL_OPTIONS = gn.GNIntegralOptions(
    quadrature_order=18,
    output_quadrature_order=3,
    coherent_accumulation=True,
    nli_prefactor=16/27,
    phase_matching_refinement=5,
)

print("FAST_OPTIONS:", FAST_OPTIONS)
print("FINAL_OPTIONS:", FINAL_OPTIONS)

FAST_OPTIONS: GNIntegralOptions(quadrature_order=8, output_quadrature_order=1, coherent_accumulation=True, nli_prefactor=0.5925925925925926, nli_correction_factor=1.0, power_definition='total_dp', phase_matching_refinement=3)
FINAL_OPTIONS: GNIntegralOptions(quadrature_order=18, output_quadrature_order=3, coherent_accumulation=True, nli_prefactor=0.5925925925925926, nli_correction_factor=1.0, power_definition='total_dp', phase_matching_refinement=5)


## 5) `GN_integral.py` 기반 maximum reach 계산 함수

방법:
1. 특정 spacing에서 `simulate_snr_curve()`로 launch power sweep
2. 각 span 수에서 best SNR(최적 launch power) 추출
3. target SNR을 만족하는 최대 span 수를 binary search

In [7]:
def make_system(spacing_ghz, modulation):
    span_length_km = 120.0 if modulation == "QPSK" else 85.0
    return gn.SystemParameters(
        channels=15,
        symbol_rate_gbd=32.0,
        spacing_ghz=float(spacing_ghz),
        span_length_km=span_length_km,
        noise_figure_db=5.0,
        transceiver_snr_db=100.0,  # line-system reach 비교이므로 TRX 제한을 사실상 제거
    )

def best_snr_for_spans(spans, spacing_ghz, fiber_name="SMF", modulation="QPSK",
                       options=FAST_OPTIONS, launch_grid_dbm=None):
    if launch_grid_dbm is None:
        launch_grid_dbm = np.arange(-6.0, 6.01, 0.25)

    system = make_system(spacing_ghz, modulation)
    fiber = FIBERS[fiber_name]

    result = gn.simulate_snr_curve(
        launch_dbm=launch_grid_dbm,
        spans=int(spans),
        fiber=fiber,
        system=system,
        options=options,
        modulation=modulation,
        roll_off=0.05,
        include_transceiver_noise=False,
    )
    idx = int(np.nanargmax(result["snr_db"]))
    return {
        "spans": int(spans),
        "distance_km": float(spans) * system.span_length_km,
        "opt_launch_dbm": float(result["launch_dbm"][idx]),
        "max_snr_db": float(result["snr_db"][idx]),
        "eta_w_inv2": float(result["eta_equivalent_w_inv2"]),
    }

def find_max_reach(spacing_ghz, fiber_name="SMF", modulation="QPSK",
                   options=FAST_OPTIONS, max_spans=160):
    target_snr_db = TARGET_SNR_DB[modulation]
    lo, hi = 1, int(max_spans)

    r1 = best_snr_for_spans(1, spacing_ghz, fiber_name, modulation, options)
    if r1["max_snr_db"] < target_snr_db:
        return {**r1, "reachable": False}

    rh = best_snr_for_spans(hi, spacing_ghz, fiber_name, modulation, options)
    if rh["max_snr_db"] >= target_snr_db:
        return {**rh, "reachable": True, "capped": True}

    while hi - lo > 1:
        mid = (lo + hi) // 2
        rm = best_snr_for_spans(mid, spacing_ghz, fiber_name, modulation, options)
        if rm["max_snr_db"] >= target_snr_db:
            lo = mid
        else:
            hi = mid

    r = best_snr_for_spans(lo, spacing_ghz, fiber_name, modulation, options)
    return {**r, "reachable": True, "capped": False}

def run_model_reach_curve(fiber_name="SMF", modulation="QPSK", options=FAST_OPTIONS):
    rows = []
    for spacing in SPACINGS_GHZ:
        print(f"running: modulation={modulation}, fiber={fiber_name}, spacing={spacing} GHz")
        r = find_max_reach(spacing, fiber_name, modulation, options=options)
        rows.append({
            "spacing_ghz": spacing,
            "model_value_spans": r["spans"],
            "model_value_km": r["distance_km"],
            "opt_launch_dbm": r["opt_launch_dbm"],
            "max_snr_db": r["max_snr_db"],
            "eta_w_inv2": r["eta_w_inv2"],
        })
    return pd.DataFrame(rows)

## 6) 단일 시나리오 실행

기본값은 **PM-QPSK / SMF / paper GN curve** 입니다.  
원하면 `SCENARIO`만 바꿔서 다른 fiber, modulation, reference mode로 실행하세요.

In [8]:
SCENARIO = {
    "modulation": "QPSK",   # "QPSK" or "16QAM"
    "fiber": "SMF",         # "PSCF", "SMF", "NZDSF"
    "paper_mode": "gn",     # "gn" or "sim"
}

print(SCENARIO)

paper_df = paper_series_to_df(
    modulation=SCENARIO["modulation"],
    fiber=SCENARIO["fiber"],
    mode=SCENARIO["paper_mode"]
)
display(paper_df)

RUN_MODEL = False  # 최종 계산 시 True
if RUN_MODEL:
    model_df = run_model_reach_curve(
        fiber_name=SCENARIO["fiber"],
        modulation=SCENARIO["modulation"],
        options=FINAL_OPTIONS,
    )
    display(model_df)
else:
    model_df = pd.DataFrame()
    print("RUN_MODEL=False: 빠른 편집용 상태입니다. 최종 검증 시 True로 바꾸세요.")

{'modulation': 'QPSK', 'fiber': 'SMF', 'paper_mode': 'gn'}


,spacing_ghz,paper_value_spans
0,33.6,37
1,35.0,41
2,40.0,50
3,45.0,58
4,50.0,66


RUN_MODEL=False: 빠른 편집용 상태입니다. 최종 검증 시 True로 바꾸세요.


## 7) 오차율 계산

정의:
\[
\mathrm{APE}_i[\%] = 100\cdot\frac{|y_{model,i}-y_{paper,i}|}{|y_{paper,i}|}
\]
요약 지표:
- point별 오차율
- MAPE
- RMSE
- Mean signed error
- Max absolute percentage error

In [9]:
def compare_model_to_paper(model_df, paper_df):
    comp = pd.merge(model_df, paper_df, on="spacing_ghz", how="inner").copy()
    comp["signed_error_spans"] = comp["model_value_spans"] - comp["paper_value_spans"]
    comp["abs_error_spans"] = np.abs(comp["signed_error_spans"])
    comp["abs_error_pct"] = 100.0 * comp["abs_error_spans"] / np.abs(comp["paper_value_spans"])
    metrics = {
        "N_points": int(len(comp)),
        "MAPE_pct": float(comp["abs_error_pct"].mean()),
        "RMSE_spans": float(np.sqrt(np.mean(comp["signed_error_spans"]**2))),
        "Mean_signed_error_spans": float(comp["signed_error_spans"].mean()),
        "Max_abs_error_pct": float(comp["abs_error_pct"].max()),
    }
    return comp, metrics

if not model_df.empty:
    comparison_df, metrics = compare_model_to_paper(model_df, paper_df)
    display(comparison_df)
    display(pd.DataFrame([metrics]))
else:
    comparison_df = pd.DataFrame()
    metrics = {}
    print("먼저 RUN_MODEL=True로 모델 계산을 수행하세요.")

먼저 RUN_MODEL=True로 모델 계산을 수행하세요.


## 8) 한 그래프에 동시 비교

요청하신 핵심 출력입니다.

In [10]:
if not comparison_df.empty:
    plt.figure(figsize=(9,6))
    plt.plot(
        comparison_df["spacing_ghz"], comparison_df["paper_value_spans"],
        marker="o", linewidth=2, label=f"Paper estimated ({SCENARIO['paper_mode']})"
    )
    plt.plot(
        comparison_df["spacing_ghz"], comparison_df["model_value_spans"],
        marker="s", linewidth=2, linestyle="--", label="GN_integral.py"
    )
    plt.xlabel("Channel spacing [GHz]")
    plt.ylabel("Maximum reach [Nspan]")
    plt.title(f"Carena Fig.5 comparison — {SCENARIO['modulation']} / {SCENARIO['fiber']}")
    plt.grid(True, alpha=0.3)
    plt.legend()
    txt = (
        f"MAPE = {metrics['MAPE_pct']:.2f}%\n"
        f"RMSE = {metrics['RMSE_spans']:.2f} spans\n"
        f"Max err = {metrics['Max_abs_error_pct']:.2f}%"
    )
    plt.text(0.02, 0.98, txt, transform=plt.gca().transAxes,
             va="top", ha="left", bbox=dict(boxstyle="round", alpha=0.15))
    plt.show()
else:
    print("comparison_df가 비어 있습니다.")

comparison_df가 비어 있습니다.


## 9) 여러 시나리오를 한 번에 비교

원문 Fig. 5는 modulation/fiber 조합이 여러 개이므로,
아래 셀은 6개 기본 시나리오를 자동 반복하여 결과를 표로 모읍니다.

In [11]:
def batch_validate_all(paper_mode="gn", options=FINAL_OPTIONS):
    rows = []
    for modulation in ["QPSK", "16QAM"]:
        for fiber_name in ["PSCF", "SMF", "NZDSF"]:
            print("="*72)
            print("Running:", modulation, fiber_name, paper_mode)
            paper_df = paper_series_to_df(modulation, fiber_name, paper_mode)
            model_df = run_model_reach_curve(fiber_name, modulation, options=options)
            comp, met = compare_model_to_paper(model_df, paper_df)
            row = {
                "modulation": modulation,
                "fiber": fiber_name,
                "paper_mode": paper_mode,
                **met
            }
            rows.append(row)
    return pd.DataFrame(rows)

RUN_BATCH = False
if RUN_BATCH:
    batch_result_df = batch_validate_all(paper_mode="gn", options=FINAL_OPTIONS)
    display(batch_result_df)
else:
    batch_result_df = pd.DataFrame()
    print("RUN_BATCH=False")

RUN_BATCH=False


## 10) 오차 원인 자동 코멘트

오차가 커도 그 원인이 항상 `GN_integral.py` 구현 오류라는 뜻은 아닙니다.
다음 가능성을 분리해야 합니다.

1. **digitized estimate 자체의 오차**  
   - 이번 notebook의 내장 paper 데이터는 exact raw data가 아님

2. **numerical integration 해상도 부족**  
   - quadrature order / refinement 부족

3. **GN model vs simulation 의 구조적 차이**  
   - GN model은 Gaussianity approximation 사용
   - simulation marker는 finite-constellation 특성을 반영

4. **power convention 차이**  
   - total DP power vs per-polarization power
   - 16/27 vs 8/27 혼용 여부

5. **threshold 해석 차이**  
   - BER→SNR 환산 근사
   - FEC margin / DSP penalty / implementation penalty 미반영

In [12]:
def automatic_comment(metrics, paper_mode="gn"):
    if not metrics:
        print("metrics가 없습니다.")
        return

    mape = metrics["MAPE_pct"]
    print("="*72)
    print("AUTOMATIC COMMENT")
    print("="*72)
    print(f"Comparison target : paper estimated ({paper_mode})")
    print(f"MAPE              : {mape:.2f}%")
    print(f"RMSE [spans]      : {metrics['RMSE_spans']:.2f}")
    print(f"Max abs err [%]   : {metrics['Max_abs_error_pct']:.2f}%")

    if mape <= 5:
        print("\n판정: 근사 digitized 기준에서도 5% 이내입니다.")
        print("의견: 현재 GN_integral.py는 해당 시나리오에서 Carena Fig.5 추정값을 잘 재현한다고 볼 수 있습니다.")
    elif mape <= 10:
        print("\n판정: 5~10% 수준의 차이입니다.")
        print("의견: 실질적으로는 양호하나, 다음 요인을 점검하세요:")
        print("- paper estimate digitization 오차")
        print("- BER→SNR threshold 근사 오차")
        print("- quadrature order 증가 필요 여부")
    else:
        print("\n판정: 10% 이상 차이입니다.")
        print("우선 확인할 항목:")
        print("1) 내장 digitized estimate가 조잡한지")
        print("2) PM-QPSK / PM-16QAM target BER 해석이 일치하는지")
        print("3) total-DP power + 16/27 convention이 논문과 일치하는지")
        print("4) GN analytical curve와 simulation marker를 혼동하지 않았는지")
        print("5) numerical integration order를 더 올려야 하는지")

    if paper_mode == "sim":
        print("\n추가 코멘트:")
        print("simulation marker와 비교하는 경우, GN model 특성상 reach를 보수적으로 예측하여")
        print("수 % ~ 10% 내외의 systematic gap이 생길 수 있습니다.")

if metrics:
    automatic_comment(metrics, SCENARIO["paper_mode"])

## 11) 결과 저장

생성 파일:
- `comparison_single_scenario.csv`
- `summary_single_scenario.json`
- `batch_result_summary.csv` (batch 실행 시)

In [13]:
summary = {
    "paper": "Carena et al., JLT 30(10), 1524-1539 (2012)",
    "doi": "10.1109/JLT.2012.2189198",
    "gn_integral_source": GN_URL,
    "scenario": SCENARIO,
    "target_ber": TARGET_BER,
    "target_snr_db_approx": TARGET_SNR_DB,
    "note": "paper values are manually estimated digitized values, not exact raw data",
}

if not comparison_df.empty:
    comparison_df.to_csv("comparison_single_scenario.csv", index=False)
    summary["metrics"] = metrics

with open("summary_single_scenario.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

if not batch_result_df.empty:
    batch_result_df.to_csv("batch_result_summary.csv", index=False)

print(json.dumps(summary, ensure_ascii=False, indent=2))

{
  "paper": "Carena et al., JLT 30(10), 1524-1539 (2012)",
  "doi": "10.1109/JLT.2012.2189198",
  "gn_integral_source": "https://raw.githubusercontent.com/kimheeseo/LSCNS/main/2026_KICS_Fall/GN_integral.py",
  "scenario": {
    "modulation": "QPSK",
    "fiber": "SMF",
    "paper_mode": "gn"
  },
  "target_ber": {
    "QPSK": 0.0017,
    "16QAM": 0.002
  },
  "target_snr_db_approx": {
    "QPSK": 9.334534961836878,
    "16QAM": 15.889881819746103
  },
  "note": "paper values are manually estimated digitized values, not exact raw data"
}


# 최종 결론 해석 가이드

### 1. `paper_mode="gn"` 비교
이 비교는 **Carena 논문의 analytical GN curve** 와 `GN_integral.py` 를 비교하는 것입니다.  
가장 직접적인 validation입니다.

### 2. `paper_mode="sim"` 비교
이 비교는 **simulation marker** 와 `GN_integral.py` 를 비교하는 것입니다.  
이 경우 차이는 단순한 numerical error가 아니라, **GN-model의 Gaussianity approximation에 의한 구조적 편차**까지 포함합니다.

### 3. 이번 notebook의 한계
- 원문 raw table 부재
- Fig.5 built-in 데이터는 수동 추정값
- BER→SNR threshold는 근사식 기반
- 따라서 본 결과는 **정밀 최종판**이 아니라 **실무용 1차 검증판**입니다.

### 4. 실무적으로 추천하는 사용법
1. 먼저 이 notebook으로 대략적인 오차 범위를 본다.
2. 사용자가 나중에 Fig.5 실제 이미지 또는 WebPlotDigitizer CSV를 확보하면,
   내장 `paper_estimated_*`를 그 CSV로 교체한다.
3. 그 후 batch validation 결과를 최종 보고서에 사용한다.